In [ ]:
import pandas as pd

import requests
import zipfile
import io

df = pd.read_csv("../data/transit_delays.csv") # reads the csv file into a pandas dataframe

df.isnull().sum()
df['stop_id'].nunique()

df = df.drop(columns = ['delay_seconds']) # drops the whole column
df = df.dropna(subset = ['route_id', 'stop_id']) # drops null values in these columns
df['timestamp'] = pd.to_datetime(df['timestamp']) # converts the timestamp column to datetime format
df.duplicated().sum()

GTFS_URL = "https://data.calgary.ca/download/npk7-z3bj/application%2Fx-zip-compressed" # stores the GTFS data url in a variable
response = requests.get(GTFS_URL)
z = zipfile.ZipFile(io.BytesIO(response.content))
stop_times = pd.read_csv(z.open('stop_times.txt')) # fetches the stop_times.txt file from the GTFS data and reads it into a pandas dataframe
df['stop_id'] = df['stop_id'].astype(int) # converts the stop_id column to integer type

hours = stop_times['arrival_time'].str.split(':').str[0].astype(int)
hours[hours >= 24].value_counts().sort_index()

def parse_gtfs_time(time_str):
    hours, minutes, seconds = map(int, time_str.split(':'))
    return pd.Timedelta(hours=hours, minutes=minutes, seconds=seconds)
stop_times['arrival_time'] = stop_times['arrival_time'].apply(parse_gtfs_time) 
stop_times_deduped = stop_times.drop_duplicates(subset=['trip_id', 'stop_id'], keep='first') # drops duplicates in the stop_times dataframe based on trip_id and stop_id columns

df['timestamp'] = df['timestamp'].dt.tz_convert('America/Edmonton')
df['timestamp'].head()

merged_df = df.merge(stop_times_deduped[['trip_id', 'stop_id', 'arrival_time']], on=['stop_id', 'trip_id'], how='left')
merged_df.head()

stop_times_deduped.duplicated(subset = ['trip_id', 'stop_id']).sum()

merged_df['arrival_time'].isnull().sum()
merged_df.shape[0]

unmatched = merged_df[merged_df['arrival_time'].isnull()]
sample_trip = unmatched.iloc[0]['trip_id']
sample_stop = unmatched.iloc[0]['stop_id']

print("Stop your scraper recorded:", sample_stop)
print("Stops actually on this trip:", stop_times[stop_times['trip_id'] == sample_trip]['stop_id'].tolist())

merged_df_cleaned = merged_df.dropna(subset=['arrival_time'])
merged_df_cleaned.shape[0]

def delay_calc(val):
    if pd.isnull(val['arrival_time']):
        return None
    time = val['timestamp'].normalize() # normalize the timestamp to remove date info
    scheduled_time = time + val['arrival_time'] # add the arrival_time to the normalized timestamp
    delay = (val['timestamp'] - scheduled_time).total_seconds() # calculate the delay
    return delay
merged_df_cleaned['delay_seconds'] = merged_df_cleaned.apply(delay_calc, axis=1) # apply the delay_calc function to each row of the merged_df_cleaned dataframe

merged_df_cleaned['delay_minutes'] = merged_df_cleaned['delay_seconds'] / 60
merged_df_cleaned['delay_minutes'] = merged_df_cleaned['delay_minutes'].round(2) # round the delay_minutes column to 2 decimal places
merged_df_cleaned['delay_minutes'] = merged_df_cleaned['delay_minutes'][(merged_df_cleaned['delay_minutes'] >= -30) & (merged_df_cleaned['delay_minutes'] <= 60)]
merged_df_cleaned = merged_df_cleaned.dropna(subset=['delay_minutes']) # drop rows with null values in the delay_minutes column
merged_df_cleaned.head()

merged_df_cleaned = merged_df_cleaned.drop(columns = ['schedule_relationship'])

merged_df_cleaned.to_csv("../data/cleaned_transit_delays.csv", index=False) # saves the cleaned dataframe to a csv file


    
    
    






Stop your scraper recorded: 7342
Stops actually on this trip: [7570, 5719, 7571, 7572, 7623, 7574, 7575, 7576, 7518, 7519, 6032, 7520, 7521, 7522, 6033, 7527, 6034, 6035, 7398, 7399, 6036, 7355, 7356, 6070, 6037, 7357, 7723, 2276, 2277, 2278, 2279, 9068, 9069, 2416, 9071, 7361, 7362, 7542, 6682, 6880, 6039, 5156, 7535, 7536, 7537, 5157, 5579]
